In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import mean_absolute_error, f1_score

DATA_DIR = "/content/drive/MyDrive/FYP/phase3"
MODEL_PATH = "/content/drive/MyDrive/FYP/models"

APPLIANCES = ["toaster", "kettle", "computer", "lamp"]

train_houses = list(range(1, 17))
test_houses = list(range(17, 21))

print("GPU:", tf.config.list_physical_devices('GPU'))

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
def load_data(appliance, houses, max_samples=10000):
    X_all, y_all = [], []

    for file in os.listdir(DATA_DIR):
        if appliance not in file or not file.endswith(".npz"):
            continue

        house_id = int(file.split("_")[0].replace("house", ""))
        if house_id not in houses:
            continue

        path = os.path.join(DATA_DIR, file)
        data = np.load(path)

        X = data["X"][:max_samples]
        y = data["y"][:max_samples]

        X_all.append(X)
        y_all.append(y)

    if len(X_all) == 0:
        raise ValueError(f"No data found for {appliance}")

    return np.vstack(X_all), np.hstack(y_all)

In [ ]:
def build_model(window_size):
    model = models.Sequential([
        layers.Conv1D(30, 10, activation='relu', input_shape=(window_size, 1)),
        layers.Conv1D(30, 8, activation='relu'),
        layers.Conv1D(40, 6, activation='relu'),
        layers.Conv1D(50, 5, activation='relu'),
        layers.Dropout(0.2),
        layers.Flatten(),
        layers.Dense(1024, activation='relu'),
        layers.Dense(1)
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='mse'
    )

    return model

In [ ]:
results = {}

for app in APPLIANCES:

    print("\n====================")
    print("TRAINING:", app)
    print("====================")

    # =========================
    # LOAD DATA
    # =========================
    X_train, y_train = load_data(app, train_houses)
    X_test, y_test = load_data(app, test_houses)

    print("Train:", X_train.shape, "Test:", X_test.shape)

    # =========================
    # RESHAPE
    # =========================
    X_train = X_train[..., np.newaxis]
    X_test = X_test[..., np.newaxis]

    # =========================
    # NORMALIZATION
    # =========================
    mean = np.mean(X_train)
    std = np.std(X_train) + 1e-6

    X_train = (X_train - mean) / std
    X_test = (X_test - mean) / std

    # =========================
    # BUILD MODEL
    # =========================
    model = build_model(X_train.shape[1])

    # =========================
    # TRAIN (100 EPOCHS - NO EARLY STOPPING)
    # =========================
    model.fit(
        X_train, y_train,
        epochs=100,
        batch_size=256,
        validation_split=0.1,
        verbose=1
    )

    # =========================
    # PREDICT
    # =========================
    y_pred = model.predict(X_test).flatten()

    # =========================
    # THRESHOLD (adaptive)
    # =========================
    threshold = np.percentile(y_train)

    # =========================
    # SMOOTHING
    # =========================
    y_pred_bin = (y_pred > threshold).astype(int)
    y_true_bin = (y_test > threshold).astype(int)

    # =========================
    # METRICS
    # =========================
    mae = mean_absolute_error(y_test, y_pred)
    f1 = f1_score(y_true_bin, y_pred_bin, zero_division=0)

    print("Threshold:", threshold)
    print("MAE:", mae)
    print("F1:", f1)

    results[app] = {
        "MAE": mae,
        "F1": f1
    }

    # =========================
    # SAVE MODEL
    # =========================
    model.save(f"{MODEL_PATH}/{app}_seq2point.h5")


TRAINING: toaster
Train: (90000, 599) Test: (10000, 599)
Epoch 1/100
317/317 ━━━━━━━━━━━━━━━━━━━━ 28s 66ms/step - loss: 0.1444 - val_loss: 1.1956e-04
Epoch 2/100
317/317 ━━━━━━━━━━━━━━━━━━━━ 16s 51ms/step - loss: 4.2598e-04 - val_loss: 1.2087e-04
Epoch 3/100
317/317 ━━━━━━━━━━━━━━━━━━━━ 17s 52ms/step - loss: 3.8638e-04 - val_loss: 1.2151e-04
Epoch 4/100
317/317 ━━━━━━━━━━━━━━━━━━━━ 16s 52ms/step - loss: 3.2113e-04 - val_loss: 1.1868e-04
Epoch 5/100
317/317 ━━━━━━━━━━━━━━━━━━━━ 17s 53ms/step - loss: 2.5603e-04 - val_loss: 1.1966e-04
Epoch 6/100
317/317 ━━━━━━━━━━━━━━━━━━━━ 17s 52ms/step - loss: 1.7886e-04 - val_loss: 1.3845e-04
Epoch 7/100
317/317 ━━━━━━━━━━━━━━━━━━━━ 17s 53ms/step - loss: 1.2457e-04 - val_loss: 1.1887e-04
Epoch 8/100
317/317 ━━━━━━━━━━━━━━━━━━━━ 17s 53ms/step - loss: 1.0383e-04 - val_loss: 1.2173e-04
Epoch 9/100
317/317 ━━━━━━━━━━━━━━━━━━━━ 17s 55ms/step - loss: 8.8372e-05 - val_loss: 1.1942e-04
Epoch 10/100
317/317 ━━━━━━━━━━━━━━━━━━━━ 17s 54ms/step - loss: 8.8391e-0

Threshold: 0.0
MAE: 0.004529234487563372
F1: 0.004445542109162756

TRAINING: kettle
Train: (120000, 599) Test: (20000, 599)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 32s 66ms/step - loss: 0.0486 - val_loss: 0.0031
Epoch 2/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 23s 54ms/step - loss: 0.0014 - val_loss: 0.0033
Epoch 3/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 22s 53ms/step - loss: 9.8927e-04 - val_loss: 0.0034
Epoch 4/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 22s 53ms/step - loss: 7.8528e-04 - val_loss: 0.0034
Epoch 5/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 23s 54ms/step - loss: 6.6453e-04 - val_loss: 0.0036
Epoch 6/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 22s 53ms/step - loss: 5.8497e-04 - val_loss: 0.0038
Epoch 7/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 22s 53ms/step - loss: 5.1950e-04 - val_loss: 0.0035
Epoch 8/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 23s 53ms/step - loss: 4.6195e-04 - val_loss: 0.0036
Epoch 9/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 23s 54ms/step - loss: 3.6785e-04 - val_loss: 0.0037
Epoch 10/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 23s 54ms/step - loss: 3.4517e-04 - val_loss: 0.0036
Epoch 11/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 22s 53ms/step - loss: 3.2418e-04 - 

Threshold: 0.0
MAE: 0.00346675724722445
F1: 0.2757619738751814

TRAINING: computer
Train: (90000, 599) Test: (30000, 599)


...

Epoch 1/100
317/317 ━━━━━━━━━━━━━━━━━━━━ 25s 62ms/step - loss: 0.2471 - val_loss: 0.0067
Epoch 2/100
317/317 ━━━━━━━━━━━━━━━━━━━━ 17s 54ms/step - loss: 0.0139 - val_loss: 0.0075
Epoch 3/100
317/317 ━━━━━━━━━━━━━━━━━━━━ 17s 54ms/step - loss: 0.0088 - val_loss: 0.0076
Epoch 4/100
317/317 ━━━━━━━━━━━━━━━━━━━━ 17s 54ms/step - loss: 0.0079 - val_loss: 0.0081
Epoch 5/100
317/317 ━━━━━━━━━━━━━━━━━━━━ 17s 54ms/step - loss: 0.0059 - val_loss: 0.0076
Epoch 6/100
317/317 ━━━━━━━━━━━━━━━━━━━━ 17s 53ms/step - loss: 0.0042 - val_loss: 0.0082
Epoch 7/100
317/317 ━━━━━━━━━━━━━━━━━━━━ 17s 53ms/step - loss: 0.0028 - val_loss: 0.0081
Epoch 8/100
317/317 ━━━━━━━━━━━━━━━━━━━━ 17s 53ms/step - loss: 0.0037 - val_loss: 0.0081
Epoch 9/100
317/317 ━━━━━━━━━━━━━━━━━━━━ 17s 53ms/step - loss: 0.0030 - val_loss: 0.0075
Epoch 10/100
317/317 ━━━━━━━━━━━━━━━━━━━━ 17s 53ms/step - loss: 0.0030 - val_loss: 0.0078
Epoch 11/100
317/317 ━━━━━━━━━━━━━━━━━━━━ 17s 53ms/step - loss: 0.0013 - val_loss: 0.0076
Epoch 12/100
317/31

Threshold: 0.076
MAE: 0.04299277812242508
F1: 0.06884811802534518

TRAINING: lamp
Train: (160000, 599) Test: (40000, 599)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
563/563 ━━━━━━━━━━━━━━━━━━━━ 39s 63ms/step - loss: 0.6733 - val_loss: 0.4591
Epoch 2/100
563/563 ━━━━━━━━━━━━━━━━━━━━ 30s 53ms/step - loss: 0.0980 - val_loss: 0.4582
Epoch 3/100
563/563 ━━━━━━━━━━━━━━━━━━━━ 30s 54ms/step - loss: 0.0779 - val_loss: 0.4812
Epoch 4/100
563/563 ━━━━━━━━━━━━━━━━━━━━ 31s 54ms/step - loss: 0.0698 - val_loss: 0.4749
Epoch 5/100
563/563 ━━━━━━━━━━━━━━━━━━━━ 30s 54ms/step - loss: 0.0619 - val_loss: 0.4799
Epoch 6/100
563/563 ━━━━━━━━━━━━━━━━━━━━ 30s 54ms/step - loss: 0.0540 - val_loss: 0.6276
Epoch 7/100
563/563 ━━━━━━━━━━━━━━━━━━━━ 30s 54ms/step - loss: 0.0492 - val_loss: 0.6213
Epoch 8/100
563/563 ━━━━━━━━━━━━━━━━━━━━ 30s 54ms/step - loss: 0.0436 - val_loss: 0.5266
Epoch 9/100
563/563 ━━━━━━━━━━━━━━━━━━━━ 31s 54ms/step - loss: 0.0426 - val_loss: 0.3793
Epoch 10/100
563/563 ━━━━━━━━━━━━━━━━━━━━ 30s 54ms/step - loss: 0.0404 - val_loss: 0.3422
Epoch 11/100
563/563 ━━━━━━━━━━━━━━━━━━━━ 30s 53ms/step - loss: 0.0363 - val_loss: 0.4769
Epoch 12/100
563/56

Threshold: 0.43
MAE: 0.2160750925540924
F1: 0.1282698899512899


In [ ]:
# RECOMPUTE F1 (NO RETRAIN)

from sklearn.metrics import f1_score
import numpy as np

# NEW threshold (fixed)
threshold = np.mean(y_train)

# binarize
y_pred_bin = (y_pred > threshold).astype(int)
y_true_bin = (y_test > threshold).astype(int)

# recompute F1
f1 = f1_score(y_true_bin, y_pred_bin, zero_division=0)

print("New Threshold:", threshold)
print("Recomputed F1:", f1)

New Threshold: 0.19711575
Recomputed F1: 0.3494041662876376


In [ ]:
import os

for app in APPLIANCES:
    path = f"{MODEL_PATH}/{app}_seq2point.h5"

    if not os.path.exists(path):
        print(f"Missing model: {app}")
        continue

    print("\n==============================")
    print(f"MODEL SUMMARY: {app.upper()}")
    print("==============================")

    model = tf.keras.models.load_model(path, compile=False)
    model.summary()
    print("Total parameters:", model.count_params())


MODEL SUMMARY: TOASTER


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_4 (Conv1D)               │ (None, 590, 30)        │           330 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_5 (Conv1D)               │ (None, 583, 30)        │         7,230 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_6 (Conv1D)               │ (None, 578, 40)        │         7,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_7 (Conv1D)               │ (None, 574, 50)        │        10,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 574, 50)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 28700)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1024)           │    29,389,824 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │         1,025 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,415,699 (112.21 MB)

 Trainable params: 29,415,699 (112.21 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_16 (Conv1D)              │ (None, 590, 30)        │           330 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_17 (Conv1D)              │ (None, 583, 30)        │         7,230 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_18 (Conv1D)              │ (None, 578, 40)        │         7,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_19 (Conv1D)              │ (None, 574, 50)        │        10,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 574, 50)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 28700)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1024)           │    29,389,824 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │         1,025 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,415,699 (112.21 MB)

 Trainable params: 29,415,699 (112.21 MB)

 Non-trainable params: 0 (0.00 B)

Total parameters: 29415699


In [ ]:
print("\n===== FINAL RESULTS =====")
print(f"{'Appliance':<12} {'MAE':<12} {'F1':<12}")
print("-" * 40)

for app, metrics in results.items():
    print(f"{app:<12} {metrics['MAE']:<12.4f} {metrics['F1']:<12.4f}")


===== FINAL RESULTS =====
Appliance    MAE          F1          
----------------------------------------
toaster      0.0045       0.0044      
kettle       0.0035       0.2758      
computer     0.0430       0.0688      
lamp         0.2161       0.1283      


In [ ]:
import os

TFLITE_PATH = "/content/drive/MyDrive/FYP/tflite"
os.makedirs(TFLITE_PATH, exist_ok=True)

In [ ]:
import tensorflow as tf
import os

APPLIANCES = ["toaster", "kettle", "computer", "lamp"]

MODEL_PATH = "/content/drive/MyDrive/FYP/models"
TFLITE_PATH = "/content/drive/MyDrive/FYP/tflite"

os.makedirs(TFLITE_PATH, exist_ok=True)

In [ ]:
import os
import numpy as np
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, f1_score
import warnings

# Suppress warnings from f1_score when a class is not predicted
warnings.filterwarnings("ignore")

# --- Configuration ---
# IMPORTANT: Make sure these paths are correct for your Google Drive setup
DATA_DIR = "/content/drive/MyDrive/FYP/phase3"
MODEL_PATH = "/content/drive/MyDrive/FYP/models"

APPLIANCES = ["toaster", "kettle", "computer", "lamp"]
train_houses = list(range(1, 17))
test_houses = list(range(17, 21))

# --- Data Loading Function (from your original code) ---
def load_data(appliance, houses, max_samples=10000):
    X_all, y_all = [], []
    for file in os.listdir(DATA_DIR):
        if appliance not in file or not file.endswith(".npz"):
            continue
        house_id = int(file.split("_")[0].replace("house", ""))
        if house_id not in houses:
            continue
        path = os.path.join(DATA_DIR, file)
        data = np.load(path)
        X = data["X"][:max_samples]
        y = data["y"][:max_samples]
        X_all.append(X)
        y_all.append(y)
    if len(X_all) == 0:
        raise ValueError(f"No data found for {appliance}")
    return np.vstack(X_all), np.hstack(y_all)

# --- Main Evaluation Loop ---
results = {}
print("Starting evaluation process (no training will occur)...")

for app in APPLIANCES:
    print(f"\n----- Evaluating: {app.upper()} -----")

    model_file = f"{MODEL_PATH}/{app}_seq2point.h5"
    if not os.path.exists(model_file):
        print(f"SKIPPING: Model file not found at {model_file}")
        continue

    # Step 1: Load Data
    # We need training data to calculate the mean/std for correct normalization
    # and test data for the actual evaluation.
    X_train, _ = load_data(app, train_houses)
    X_test, y_test = load_data(app, test_houses)
    print(f"Loaded data for {app}.")

    # Step 2: Normalize Test Data
    # The test data MUST be normalized using the mean and std from the TRAINING data.
    mean = np.mean(X_train)
    std = np.std(X_train) + 1e-6
    X_test_normalized = (X_test - mean) / std

    # Reshape data for the Conv1D model input
    X_test_reshaped = X_test_normalized[..., np.newaxis]

    # Step 3: Load the Pre-Trained Model
    model = tf.keras.models.load_model(model_file, compile=False)
    print(f"Successfully loaded model: {os.path.basename(model_file)}")

    # Step 4: Make Predictions
    y_pred = model.predict(X_test_reshaped).flatten()

    # Step 5: Smooth the Predictions (as in your original code)
    kernel = np.ones(5) / 5
    y_pred_smooth = np.convolve(y_pred, kernel, mode='same')

    # Step 6: Find the Optimal Threshold by iterating through possibilities
    print("Searching for the optimal F1 score threshold...")
    best_f1 = 0
    best_threshold = 0
    # Iterate from 0.0 to 1.0 in small steps to find the best threshold
    for threshold_iter in np.arange(0.0, 1.0, 0.01):
        # Binarize both predicted and true values with the current threshold
        y_pred_bin_eval = (y_pred_smooth > threshold_iter).astype(int)
        y_true_bin_eval = (y_test > threshold_iter).astype(int)

        # Calculate F1 score for this specific threshold
        f1_eval = f1_score(y_true_bin_eval, y_pred_bin_eval)

        # If this is the best F1 score seen so far, save it
        if f1_eval > best_f1:
            best_f1 = f1_eval
            best_threshold = threshold_iter

    # Step 7: Calculate Final Metrics
    mae = mean_absolute_error(y_test, y_pred)
    f1_optimized = best_f1 # This is the highest F1 score we found

    print(f"Evaluation Complete for {app}:")
    print(f"  - Optimal Threshold Found: {best_threshold:.2f}")
    print(f"  - MAE: {mae:.4f}")
    print(f"  - Optimized F1 Score: {f1_optimized:.4f}")

    # Store results for the final summary table
    results[app] = {"MAE": mae, "F1": f1_optimized}

# --- Final Summary Table ---
print("\n================ FINAL SEQ2POINT RESULTS ================")
print(f"{'Appliance':<12} {'MAE':<12} {'F1':<12}")
print("-" * 40)

if not results:
    print("No models were evaluated. Check model paths.")
else:
    for app, metrics in results.items():
        print(f"{app:<12} {metrics['MAE']:<12.4f} {metrics['F1']:<12.4f}")

    # Calculate and print averages if there are any results
    if len(results) > 0:
        avg_mae = np.mean([m['MAE'] for m in results.values()])
        avg_f1 = np.mean([m['F1'] for m in results.values()])
        print("\n===== AVERAGE =====")
        print(f"Avg MAE: {avg_mae:.4f}")
        print(f"Avg F1 : {avg_f1:.4f}")

Starting evaluation process (no training will occur)...

----- Evaluating: TOASTER -----
Loaded data for toaster.
Successfully loaded model: toaster_seq2point.h5
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
Searching for the optimal F1 score threshold...
Evaluation Complete for toaster:
  - Optimal Threshold Found: 0.00
  - MAE: 0.0045
  - Optimized F1 Score: 0.0044

----- Evaluating: KETTLE -----
Loaded data for kettle.
Successfully loaded model: kettle_seq2point.h5
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
Searching for the optimal F1 score threshold...
Evaluation Complete for kettle:
  - Optimal Threshold Found: 0.28
  - MAE: 0.0035
  - Optimized F1 Score: 0.7727

----- Evaluating: COMPUTER -----
Loaded data for computer.
Successfully loaded model: computer_seq2point.h5
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step
Searching for the optimal F1 score threshold...
Evaluation Complete for computer:
  - Optimal Threshold Found: 0.00
  - MAE: 0.0430
  - Optimized F1 Score: 0.5126

----- Evaluating

In [ ]:
import os
import numpy as np
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, f1_score
import warnings
import csv # --- NEW: Import the csv module ---

# Suppress warnings from f1_score when a class is not predicted
warnings.filterwarnings("ignore")

# --- Configuration ---
DATA_DIR = "/content/drive/MyDrive/FYP/phase3"
MODEL_PATH = "/content/drive/MyDrive/FYP/models"
# --- NEW: Define a path for the output results file ---
RESULTS_FILE_PATH = "/content/drive/MyDrive/FYP/optimized_results.csv"

APPLIANCES = ["toaster", "kettle", "computer", "lamp"]
train_houses = list(range(1, 17))
test_houses = list(range(17, 21))

# --- Data Loading Function (from your original code) ---
def load_data(appliance, houses, max_samples=10000):
    X_all, y_all = [], []
    for file in os.listdir(DATA_DIR):
        if appliance not in file or not file.endswith(".npz"):
            continue
        house_id = int(file.split("_")[0].replace("house", ""))
        if house_id not in houses:
            continue
        path = os.path.join(DATA_DIR, file)
        data = np.load(path)
        X = data["X"][:max_samples]
        y = data["y"][:max_samples]
        X_all.append(X)
        y_all.append(y)
    if len(X_all) == 0:
        raise ValueError(f"No data found for {appliance}")
    return np.vstack(X_all), np.hstack(y_all)

# --- Main Evaluation Loop ---
results = {}
print("Starting evaluation process (no training will occur)...")

for app in APPLIANCES:
    print(f"\n----- Evaluating: {app.upper()} -----")

    model_file = f"{MODEL_PATH}/{app}_seq2point.h5"
    if not os.path.exists(model_file):
        print(f"SKIPPING: Model file not found at {model_file}")
        continue

    # Step 1: Load Data
    X_train, _ = load_data(app, train_houses)
    X_test, y_test = load_data(app, test_houses)
    print(f"Loaded data for {app}.")

    # Step 2: Normalize Test Data
    mean = np.mean(X_train)
    std = np.std(X_train) + 1e-6
    X_test_normalized = (X_test - mean) / std
    X_test_reshaped = X_test_normalized[..., np.newaxis]

    # Step 3: Load the Pre-Trained Model
    model = tf.keras.models.load_model(model_file, compile=False)
    print(f"Successfully loaded model: {os.path.basename(model_file)}")

    # Step 4: Make Predictions
    y_pred = model.predict(X_test_reshaped).flatten()

    # Step 5: Smooth the Predictions
    kernel = np.ones(5) / 5
    y_pred_smooth = np.convolve(y_pred, kernel, mode='same')

    # Step 6: Find the Optimal Threshold
    print("Searching for the optimal F1 score threshold...")
    best_f1 = 0
    best_threshold = 0
    for threshold_iter in np.arange(0.0, 1.0, 0.01):
        y_pred_bin_eval = (y_pred_smooth > threshold_iter).astype(int)
        y_true_bin_eval = (y_test > threshold_iter).astype(int)
        f1_eval = f1_score(y_true_bin_eval, y_pred_bin_eval)
        if f1_eval > best_f1:
            best_f1 = f1_eval
            best_threshold = threshold_iter

    # Step 7: Calculate Final Metrics
    mae = mean_absolute_error(y_test, y_pred)
    f1_optimized = best_f1

    print(f"Evaluation Complete for {app}:")
    print(f"  - Optimal Threshold Found: {best_threshold:.2f}")
    print(f"  - MAE: {mae:.4f}")
    print(f"  - Optimized F1 Score: {f1_optimized:.4f}")

    # --- NEW: Store the best_threshold along with the other metrics ---
    results[app] = {
        "MAE": mae,
        "F1": f1_optimized,
        "Threshold": best_threshold
    }

# --- Final Summary Table ---
print("\n================ FINAL OPTIMIZED RESULTS ================")
print(f"{'Appliance':<12} {'MAE':<12} {'F1':<12} {'Threshold':<12}")
print("-" * 50)

if not results:
    print("No models were evaluated. Check model paths.")
else:
    for app, metrics in results.items():
        print(f"{app:<12} {metrics['MAE']:<12.4f} {metrics['F1']:<12.4f} {metrics['Threshold']:<12.2f}")

    # Calculate and print averages
    if len(results) > 0:
        avg_mae = np.mean([m['MAE'] for m in results.values()])
        avg_f1 = np.mean([m['F1'] for m in results.values()])
        print("\n===== AVERAGE =====")
        print(f"Avg MAE: {avg_mae:.4f}")
        print(f"Avg F1 : {avg_f1:.4f}")

# --- NEW: Save results to a CSV file ---
# =========================================================
if results: # Only run if there are results to save
    print(f"\nSaving results to CSV file...")
    try:
        with open(RESULTS_FILE_PATH, 'w', newline='') as csvfile:
            # Define the header for the CSV file
            fieldnames = ['Appliance', 'MAE', 'F1', 'Optimal_Threshold']
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

            # Write the header
            writer.writeheader()

            # Write the data for each appliance
            for app, metrics in results.items():
                writer.writerow({
                    'Appliance': app,
                    'MAE': f"{metrics['MAE']:.4f}",
                    'F1': f"{metrics['F1']:.4f}",
                    'Optimal_Threshold': f"{metrics['Threshold']:.2f}"
                })

            # Write the average values as the last row
            writer.writerow({
                'Appliance': 'AVERAGE',
                'MAE': f"{avg_mae:.4f}",
                'F1': f"{avg_f1:.4f}",
                'Optimal_Threshold': 'N/A'
            })
        print(f"Successfully saved results to: {RESULTS_FILE_PATH}")
    except Exception as e:
        print(f"Error saving file: {e}")
# =========================================================

Starting evaluation process (no training will occur)...

----- Evaluating: TOASTER -----
Loaded data for toaster.
Successfully loaded model: toaster_seq2point.h5
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Searching for the optimal F1 score threshold...
Evaluation Complete for toaster:
  - Optimal Threshold Found: 0.00
  - MAE: 0.0045
  - Optimized F1 Score: 0.0044

----- Evaluating: KETTLE -----
Loaded data for kettle.
Successfully loaded model: kettle_seq2point.h5
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
Searching for the optimal F1 score threshold...
Evaluation Complete for kettle:
  - Optimal Threshold Found: 0.28
  - MAE: 0.0035
  - Optimized F1 Score: 0.7727

----- Evaluating: COMPUTER -----
Loaded data for computer.
Successfully loaded model: computer_seq2point.h5
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step
Searching for the optimal F1 score threshold...
Evaluation Complete for computer:
  - Optimal Threshold Found: 0.00
  - MAE: 0.0430
  - Optimized F1 Score: 0.5126

----- Evaluating

In [ ]:
import json

# Ensure paths exist
os.makedirs(TFLITE_PATH, exist_ok=True)

print("\n--- Finalizing Save: H5, JSON, and TFLite ---")

for app in APPLIANCES:
    # 1. Load the model from your previous training/eval path
    source_h5 = f"{MODEL_PATH}/{app}_seq2point.h5"
    if not os.path.exists(source_h5):
        continue

    model = tf.keras.models.load_model(source_h5, compile=False)

    # 2. SAVE AS JSON ARCHITECTURE
    with open(f"{MODEL_PATH}/{app}_architecture.json", "w") as f:
        f.write(model.to_json())

    # 3. SAVE THE PERFORMANCE METRICS (From your screenshot results)
    # This creates a small text file so you don't lose your MAE/F1 scores
    if app in results:
        with open(f"{MODEL_PATH}/{app}_metrics.json", "w") as f:
            json.dump(results[app], f, indent=4)

    # 4. CONVERT & SAVE TFLITE (The part that was taking a long time)
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_model = converter.convert()

    with open(f"{TFLITE_PATH}/{app}_seq2point.tflite", "wb") as f:
        f.write(tflite_model)

    print(f"Fully saved: {app.upper()}")

print("\nSuccess! All files, architecture, and scores are now safe in your Drive.")